### Libraries

In [2]:
import torch
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from tqdm.auto import tqdm

In [3]:
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
print(f"Using device: {device}")

Using device: cpu


In [5]:
df = pd.read_csv("df_topics_with_labels.csv")
df

,Speech_ID,Paragraph_ID,TEXT_SPAN,role,language_of_text,name,Macro_Domain,Domain_Confidence,Sub_Issue_ID,Sub_Issue_Name
0,156344,156344_p3,La majorité de la commission estime que l'intr...,For the committee,FR,Hiltpold Hugues,"National Security, Armed Forces, Defense",0.393507,4,Military firearm retention rules
1,156344,156344_p4,A noter que la charge de travail supplémentair...,For the committee,FR,Hiltpold Hugues,"National Security, Armed Forces, Defense",0.356015,4,Military firearm retention rules
2,156344,156344_p5,"Pour la minorité de la commission, qui soutien...",For the committee,FR,Hiltpold Hugues,"National Security, Armed Forces, Defense",0.395408,4,Military firearm retention rules
3,195813,195813_p2,Zusammengefasst: Wir müssen uns einfach bewuss...,Member of the federal council,DE,Maurer Ueli,"National Security, Armed Forces, Defense",0.362294,1,Border guard personnel staffing
4,156424,156424_p0,Das Anliegen ist aus der Sicherheitspolitische...,Member of the federal council,DE,Widmer-Schlumpf Eveline,"National Security, Armed Forces, Defense",0.346319,6,Cyberdefence strategy implementation
...,...,...,...,...,...,...,...,...,...,...
1801,287479,287479_p5,Wir danken dem Bundesrat für die rasche Stellu...,For the committee,DE,Minder Thomas,"National Security, Armed Forces, Defense",0.323480,5,Rüstungsexportkontrollverordnung
1802,288036,288036_p0,Quelques mots d'abord sur le contexte de l'obj...,For the committee,FR,Addor Jean-Luc,"National Security, Armed Forces, Defense",0.379429,8,Military support for foreign representation pr...
1803,288036,288036_p2,Pourquoi cette affaire doit-elle passer devant...,For the committee,FR,Addor Jean-Luc,"National Security, Armed Forces, Defense",0.456650,8,Military support for foreign representation pr...
1804,288610,288610_p4,Die Aufgriffe des Grenzwachtkorps sind beachtl...,Member of the federal council,DE,Maurer Ueli,"National Security, Armed Forces, Defense",0.343368,1,Border guard personnel staffing


# Hypothesis Template

In [57]:
hp_df = df.copy()

In [98]:
PRO_TEMPLATE = "The speaker advocates in favor of {}."
CON_TEMPLATE = "The speaker advocates against {}."

In [99]:
hp_df["pro_hypothesis"] = hp_df["Sub_Issue_Name"].apply(
    lambda x: PRO_TEMPLATE.format(x)
)
hp_df["con_hypothesis"] = hp_df["Sub_Issue_Name"].apply(
    lambda x: CON_TEMPLATE.format(x)
)

# NLI Model

In [100]:
model_name = "MoritzLaurer/mDeBERTa-v3-base-xnli-multilingual-nli-2mil7"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)
model

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification LOAD REPORT from: MoritzLaurer/mDeBERTa-v3-base-xnli-multilingual-nli-2mil7
Key                             | Status     |  | 
--------------------------------+------------+--+-
deberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


DebertaV2ForSequenceClassification(
  (deberta): DebertaV2Model(
    (embeddings): DebertaV2Embeddings(
      (word_embeddings): Embedding(251000, 768, padding_idx=0)
      (LayerNorm): LayerNorm((768,), eps=1e-07, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): DebertaV2Encoder(
      (layer): ModuleList(
        (0-11): 12 x DebertaV2Layer(
          (attention): DebertaV2Attention(
            (self): DisentangledSelfAttention(
              (query_proj): Linear(in_features=768, out_features=768, bias=True)
              (key_proj): Linear(in_features=768, out_features=768, bias=True)
              (value_proj): Linear(in_features=768, out_features=768, bias=True)
              (pos_dropout): Dropout(p=0.1, inplace=False)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): DebertaV2SelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): Layer

In [101]:
import numpy as np

n_rows = len(hp_df)
pro_entail = np.zeros(n_rows)
con_entail = np.zeros(n_rows)

In [102]:
BATCH_SIZE = 1

model.to(device)
model.eval()

for i in tqdm(range(0, n_rows, BATCH_SIZE), desc="Processing rows"):
    batch = hp_df.iloc[i : i + BATCH_SIZE]

    text_batch = batch["TEXT_SPAN"].tolist()

    # Process both hypotheses for the same sample
    for hypothesis_col, target_array in [
        ("pro_hypothesis", pro_entail),
        ("con_hypothesis", con_entail),
    ]:
        hypothesis_batch = batch[hypothesis_col].tolist()

        encodings = tokenizer(
            text_batch,
            hypothesis_batch,
            truncation=True,
            padding=True,
            return_tensors="pt",
        ).to(device)

        with torch.no_grad():
            outputs = model(**encodings)
            probs = torch.softmax(outputs.logits, dim=1).cpu().numpy()

        entailment_score = probs[0, 0]
        target_array[i : i + BATCH_SIZE] = entailment_score

        # Print result immediately
        print("\n========================")
        print(f"Row index: {i}")
        print(f"Hypothesis column: {hypothesis_col}")
        print(f"TEXT_SPAN:\n{batch.iloc[0]['TEXT_SPAN']}\n")
        print(f"HYPOTHESIS:\n{batch.iloc[0][hypothesis_col]}\n")
        print(f"Entailment score: {entailment_score:.4f}")
        print("========================")

Processing rows:   0%|          | 0/1806 [00:00<?, ?it/s]


Row index: 0
Hypothesis column: pro_hypothesis
TEXT_SPAN:
La majorité de la commission estime que l'introduction de l'obligation de déclarer a posteriori des armes à feu qui ne sont pas encore enregistrées répond à une demande expresse des cantons. Ce sont les responsables de la sécurité dans les cantons qui font cette demande et c'est finalement eux qui sont les mieux placés pour juger de la pertinence de cette déclaration a posteriori. Celle-ci permettra d'identifier rapidement des armes en cas de vol. Elle contribuera à accroître la sécurité publique, notamment celle des agents de police qui sont sur le terrain.

HYPOTHESIS:
The speaker advocates in favor of Military firearm retention rules.

Entailment score: 0.5200

Row index: 0
Hypothesis column: con_hypothesis
TEXT_SPAN:
La majorité de la commission estime que l'introduction de l'obligation de déclarer a posteriori des armes à feu qui ne sont pas encore enregistrées répond à une demande expresse des cantons. Ce sont les respons

KeyboardInterrupt: 

In [103]:
hp_df["Support_Entail"] = pro_entail
hp_df["Oppose_Entail"] = con_entail
hp_df["Stance_Score"] = hp_df["Support_Entail"] - hp_df["Oppose_Entail"]

In [104]:
hp_df[hp_df["Stance_Score"] > 0.75]

,Speech_ID,Paragraph_ID,TEXT_SPAN,role,language_of_text,name,Macro_Domain,Domain_Confidence,Sub_Issue_ID,Sub_Issue_Name,LANG,pro_hypothesis,con_hypothesis,Support_Entail,Support_Contradict,Oppose_Entail,Stance_Score
3,195813,195813_p2,Zusammengefasst: Wir müssen uns einfach bewuss...,Member of the federal council,DE,Maurer Ueli,"National Security, Armed Forces, Defense",0.362294,1,Border guard personnel staffing,german,The speaker advocates in favor of Border guard...,The speaker advocates against Border guard per...,0.973633,0.000758,0.000555,0.973078
4,156424,156424_p0,Das Anliegen ist aus der Sicherheitspolitische...,Member of the federal council,DE,Widmer-Schlumpf Eveline,"National Security, Armed Forces, Defense",0.346319,6,Cyberdefence strategy implementation,german,The speaker advocates in favor of Cyberdefence...,The speaker advocates against Cyberdefence str...,0.996094,0.000344,0.000319,0.995775
5,156426,156426_p0,Mit der vorliegenden Motion der Sicherheitspol...,For the committee,DE,Eder Joachim,"National Security, Armed Forces, Defense",0.319427,6,Cyberdefence strategy implementation,german,The speaker advocates in favor of Cyberdefence...,The speaker advocates against Cyberdefence str...,0.922363,0.000273,0.000178,0.922186
6,195869,195869_p1,"L'initiative populaire ""pour la sécurité alime...",For the committee,FR,Lüscher Christian,"National Security, Armed Forces, Defense",0.376445,0,Domestic agricultural supply security,french,The speaker advocates in favor of Domestic agr...,The speaker advocates against Domestic agricul...,0.862793,0.006927,0.000717,0.862076


In [109]:
hp_df.iloc[6].to_dict()

{'Speech_ID': 195869,
 'Paragraph_ID': '195869_p1',
 'TEXT_SPAN': 'L\'initiative populaire "pour la sécurité alimentaire" propose de compléter la Constitution fédérale avec un article 104a intitulé "Sécurité alimentaire".',
 'role': 'For the committee',
 'language_of_text': 'FR',
 'name': 'Lüscher Christian',
 'Macro_Domain': 'National Security, Armed Forces, Defense',
 'Domain_Confidence': 0.37644467,
 'Sub_Issue_ID': 0,
 'Sub_Issue_Name': 'Domestic agricultural supply security',
 'LANG': 'french',
 'pro_hypothesis': 'The speaker advocates in favor of Domestic agricultural supply security.',
 'con_hypothesis': 'The speaker advocates against Domestic agricultural supply security.',
 'Support_Entail': 0.86279296875,
 'Support_Contradict': 0.006927490234375,
 'Oppose_Entail': 0.0007171630859375,
 'Stance_Score': 0.8620758056640625}

In [86]:
hp_df.to_csv("final_nli_results.csv", index=False)